In [27]:
!pip install ultralytics -q

In [28]:
!pip install roboflow -q

#Projede büyük bir model eğitmek için veri seti yüklediğim yer.


In [29]:
from roboflow import Roboflow
rf = Roboflow(api_key="REDACTED_API_KEY")
project = rf.workspace("slas-workspace-ofcjv").project("landslide-segmentation-zpy5e-kqanl")
version = project.version(1)
dataset = version.download("yolov11")

#Yolov11 kullanarak modeli eğitmek için çektiğim veri seti.

loading Roboflow workspace...
loading Roboflow project...


In [30]:
import torch
from ultralytics import YOLO

#Yolo kullanımı için

In [31]:
if torch.cuda.is_available():
    print(f"Kullanılan GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Belleği: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.2f} GB")
else:
    raise Exception("HATA: GPU aktif değil!")

    # Proje için GPU gerektiği için modeli eğitmeden önce GPU çalışmasını kontrol etmesi için.

Kullanılan GPU: Tesla T4
GPU Belleği: 14.56 GB


In [ ]:
import torch
print("GPU Mevcut mu?:", torch.cuda.is_available())
print("GPU Adı:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "Yok")

GPU Mevcut mu?: True
GPU Adı: Tesla T4


In [33]:
import yaml
data_config = {
    "train": "/content/dataset/train/images",
    "val": "/content/dataset/val/images",
    "test": "/content/dataset/test/images",
    "nc": 1,
    "names": ["Landslide"]
}

with open("landslide_data.yaml", "w") as f:
    yaml.dump(data_config, f)

print("landslide_data.yaml dosyası başarıyla oluşturuldu.")

landslide_data.yaml dosyası başarıyla oluşturuldu.


In [ ]:
import os

os.makedirs("/content/dataset/train/images", exist_ok=True)
os.makedirs("/content/dataset/val/images", exist_ok=True)

#Basit ve kendim oluşturduğum veri.


In [ ]:
import os

os.makedirs("/content/dataset/train/images", exist_ok=True)
os.makedirs("/content/dataset/train/labels", exist_ok=True)
os.makedirs("/content/dataset/val/images", exist_ok=True)
os.makedirs("/content/dataset/val/labels", exist_ok=True)

##Basit ve kendim oluşturduğum veri.


In [34]:
model = YOLO("yolo11m-seg.pt")

#Orta boyut veri modeli

In [35]:
yaml_yolu = dataset.location + "/data.yaml"
print(f"Kullanılacak veri haritası: {yaml_yolu}")

print("Veri haritasını kullanarak modeli eğitiyorum.")


Kullanılacak veri haritası: /content/Landslide-Segmentation-1/data.yaml
Veri haritasını kullanarak modeli eğitiyorum.


In [ ]:
results = model.train(
    data=yaml_yolu,
    epochs=100,
    imgsz=640,
    batch=16,
    device=0,
    optimizer="AdamW",
    lr0=0.0001,
    cos_lr=True,
    weight_decay=0.0005,

    fliplr=0.5,
    flipud=0.5,
    degrees=90.0,
    hsv_s=0.5,
    hsv_v=0.4
)

Ultralytics 8.4.33 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/Landslide-Segmentation-1/data.yaml, degrees=90.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.5, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.5, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0001, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11m-seg.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train4, nbs=64, nms=False, opset=None, optimize=False, optimizer=AdamW, overlap_mask=True, patie

In [ ]:

print("Model doğrulama verileri üzerinde test ediliyor...")
metrics = model.val()

print(f"Maske Segmentasyon mAP50: {metrics.seg.map50:.4f}")
print(f"Sınır Kutusu mAP50: {metrics.box.map50:.4f}")

test_image = "/content/Landslide-Segmentation-1/train/images"
inference_results = model.predict(source=test_image, conf=0.6, save=True, show_labels=True)

print("'runs/segment/predict' klasöründe çıktı sonuçlarını görüntüleyebilirim.")